# Module 10: Cluster Analysis - Complete Guide

## 1. Introduction: Unsupervised Discovery and Segmentation
You have now moved from supervised and structured modeling (like Regression, where we predict outcomes based on variance) to unsupervised discovery. 

Clustering is the art of finding hidden natural groupings within your data without the need for pre-existing labels. In the pharmaceutical and healthcare world, this is the foundation of Segmentation. Whether you are segmenting physicians by their prescribing habits or patients by their adherence patterns, clustering allows the data to speak for itself.

## 2. The Philosophy: Why Cluster?
Unlike regression (which models the relationship between X and Y) or factor analysis (which reduces variables into latent factors), Clustering organizes your observations (rows) into buckets.

- **No Prior Labels:** You are not looking for a predefined target. You are looking for internal similarities.
- **Intra-cluster Similarity:** The goal is to maximize the similarity of items WITHIN a group.
- **Inter-cluster Dissimilarity:** The goal is to maximize the difference BETWEEN different groups.

In [ ]:
# 3. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Imports for Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch

# Set display options for better readability
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for perfect reproducibility
np.random.seed(42)

print("Libraries successfully imported! Environment is ready for Cluster Analysis.")

## 4. Data Creation: Synthetic Physician Prescribing Habits
To perform cluster analysis, we need a dataset with natural groupings. We will simulate a dataset of 300 Physicians with three key features:
1. **Prescriptions_Written:** The total volume of a specific drug class prescribed (Scale: 0 to 10000).
2. **Years_Experience:** The number of years the physician has been practicing (Scale: 0 to 40).
3. **Seminar_Attendance:** Number of sponsored educational seminars attended per year (Scale: 0 to 10).

We will intentionally design 3 underlying segments (archetypes):
- Segment A: High volume, high experience, low seminar attendance (Veterans).
- Segment B: Low volume, low experience, high seminar attendance (Newbies).
- Segment C: Moderate volume, moderate experience, moderate attendance (Mainstream).

In [ ]:
n_physicians = 300

# Segment A: Veterans (n=100)
rx_a = np.random.normal(loc=8000, scale=800, size=100)
exp_a = np.random.normal(loc=30, scale=4, size=100)
sem_a = np.random.normal(loc=1, scale=1, size=100)

# Segment B: Newbies (n=100)
rx_b = np.random.normal(loc=2000, scale=500, size=100)
exp_b = np.random.normal(loc=5, scale=2, size=100)
sem_b = np.random.normal(loc=8, scale=1.5, size=100)

# Segment C: Mainstream (n=100)
rx_c = np.random.normal(loc=5000, scale=600, size=100)
exp_c = np.random.normal(loc=15, scale=3, size=100)
sem_c = np.random.normal(loc=4, scale=1.5, size=100)

# Combine and clip to valid ranges
rx = np.clip(np.concatenate([rx_a, rx_b, rx_c]), 0, 10000)
exp = np.clip(np.concatenate([exp_a, exp_b, exp_c]), 0, 45)
sem = np.clip(np.concatenate([sem_a, sem_b, sem_c]), 0, 15)

df_physicians = pd.DataFrame({
    'Prescriptions_Written': rx,
    'Years_Experience': exp,
    'Seminar_Attendance': sem
})

# Shuffle the dataset so the order doesn't hint at the clusters
df_physicians = df_physicians.sample(frac=1, random_state=42).reset_index(drop=True)

print("Synthetic Physician Dataset Created.")
print(df_physicians.head().round(1))

## 5. Initial Data Exploration
Let's look at the basic descriptive statistics. Notice the massive difference in scale between Prescriptions_Written (thousands) and Years_Experience (tens).

In [ ]:
print("--- Descriptive Statistics ---")
print(df_physicians.describe().round(2))

# Let's visualize the raw data distributions
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(df_physicians['Prescriptions_Written'], kde=True, color='blue')
plt.title('Distribution of Prescriptions')

plt.subplot(1, 3, 2)
sns.histplot(df_physicians['Years_Experience'], kde=True, color='green')
plt.title('Distribution of Experience')

plt.subplot(1, 3, 3)
sns.histplot(df_physicians['Seminar_Attendance'], kde=True, color='orange')
plt.title('Distribution of Seminars')

plt.tight_layout()
plt.show()

print("You can visually detect 'lumps' or multi-modal peaks in these histograms, hinting at underlying clusters.")

## 6. Core Concept 1: Data Preprocessing and The Scale Problem
Clustering is entirely based on Distance Metrics (how mathematically far apart two points are in space). 

**The Problem:** If we use Euclidean distance right now, the gap in Prescriptions (e.g., 5000 vs 8000 = 3000 difference) will completely overwhelm the gap in Experience (e.g., 5 vs 30 = 25 difference). The algorithm will essentially ignore experience and seminars.

**The Solution:** Normalization & Standardization (Z-scores). We must bring all variables to a common scale (mean = 0, standard deviation = 1).

In [ ]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform the data
scaled_data = scaler.fit_transform(df_physicians)

# Convert back to DataFrame for easy viewing
df_scaled = pd.DataFrame(scaled_data, columns=df_physicians.columns)

print("--- Data After Standardization (Z-Scores) ---")
print(df_scaled.head().round(3))
print("\n--- New Descriptive Statistics ---")
print(df_scaled.describe().round(2))
print("\nNotice that the mean is now 0 and the standard deviation is 1 for ALL columns.")
print("The features will now contribute equally to the distance calculations.")

## 7. Understanding Distance Metrics
Algorithms need a way to define "closeness".

- **Euclidean Distance:** The straight-line distance between two points. 
  Formula: d(p, q) = sqrt((q1 - p1)^2 + (q2 - p2)^2 + ... + (qn - pn)^2)
- **Manhattan Distance:** The distance traveling along axes at right angles (like a city grid).
  Formula: d(p, q) = |q1 - p1| + |q2 - p2| + ... + |qn - pn|

K-Means primarily uses Euclidean distance. Let's see how standardizing affects this.

In [ ]:
# Let's compare Physician 0 and Physician 1
physician_0_raw = df_physicians.iloc[0].values
physician_1_raw = df_physicians.iloc[1].values

physician_0_scaled = df_scaled.iloc[0].values
physician_1_scaled = df_scaled.iloc[1].values

# Calculate Euclidean Distance manually
dist_raw = np.sqrt(np.sum((physician_0_raw - physician_1_raw)**2))
dist_scaled = np.sqrt(np.sum((physician_0_scaled - physician_1_scaled)**2))

print("--- Distance Comparison ---")
print(f"Raw Distance: {dist_raw:.2f} (Dominated almost entirely by the Prescriptions column)")
print(f"Scaled Distance: {dist_scaled:.2f} (Balanced representation of all three traits)")

## 8. Core Concept 2: Algorithmic Approaches - K-Means
K-Means is a **Partitioning Method**. You define the number of clusters (k) upfront, and the algorithm iterates to find the optimal center points (centroids).

**How it works:**
1. Randomly place 'k' centroids in the data space.
2. Assign each data point to the nearest centroid.
3. Recalculate the position of the centroids based on the mean of all points assigned to them.
4. Repeat until the centroids stop moving (convergence).

Let's run K-Means with k=3.

In [ ]:
# Initialize KMeans with k=3
kmeans_model = KMeans(n_clusters=3, init='k-means++', max_iter=300, n_init=10, random_state=42)

# Fit the model on the SCALED data
kmeans_labels = kmeans_model.fit_predict(df_scaled)

# Attach the cluster labels back to our ORIGINAL dataframe for business interpretation
df_physicians['Cluster_Label'] = kmeans_labels

print("K-Means clustering complete!")
print("First 10 rows with Cluster Assignments:")
print(df_physicians.head(10))

## 9. Visualizing the Clusters
We have three dimensions, but we can visualize them using 2D scatter plots to see how well the algorithm separated the groups.

In [ ]:
plt.figure(figsize=(14, 6))

# Plot 1: Experience vs Prescriptions
plt.subplot(1, 2, 1)
sns.scatterplot(data=df_physicians, x='Years_Experience', y='Prescriptions_Written', 
                hue='Cluster_Label', palette='viridis', s=80, alpha=0.8)
plt.title('Physician Segments: Exp vs. Volume', fontsize=14)

# Plot 2: Seminars vs Prescriptions
plt.subplot(1, 2, 2)
sns.scatterplot(data=df_physicians, x='Seminar_Attendance', y='Prescriptions_Written', 
                hue='Cluster_Label', palette='viridis', s=80, alpha=0.8)
plt.title('Physician Segments: Education vs. Volume', fontsize=14)

plt.tight_layout()
plt.show()

print("The plots reveal perfectly distinct, non-overlapping segments.")
print("This visual separation proves the algorithm successfully maximized inter-cluster dissimilarity.")

## 10. Business Interpretation: Profiling the Segments
Finding clusters is only half the battle. You must interpret them for business strategy. 
We do this by calculating the mean of each variable for each cluster.

In [ ]:
cluster_profiles = df_physicians.groupby('Cluster_Label').mean().round(1)
print("--- Cluster Profiles (Centroids in Original Scale) ---")
print(cluster_profiles)

print("\n--- Strategic Business Interpretation ---")
print("Cluster 0: Moderate Volume (~5000), Moderate Exp (~15 yrs), Moderate Seminars (~4). -> The 'Mainstream'.")
print("Cluster 1: High Volume (~8000), High Exp (~30 yrs), Low Seminars (~1). -> The 'Veterans'. Highly lucrative but hard to reach via education.")
print("Cluster 2: Low Volume (~2000), Low Exp (~5 yrs), High Seminars (~8). -> The 'Newbies'. Low current value, but highly engaged. Prime targets for brand loyalty programs.")

## 11. Core Concept 3: Validation - The Elbow Method
In the previous step, we *knew* k=3 because we generated the data. In reality, you don't know the optimal number of clusters.

**The Elbow Method:** We run K-Means for varying values of k (e.g., 1 to 10) and calculate the Within-Cluster Sum of Squares (WCSS), also known as inertia. WCSS measures how tightly packed the clusters are.

As k increases, WCSS naturally drops. We look for the "elbow" or kink in the curve where adding more clusters yields diminishing returns.

In [ ]:
wcss = []
k_range = range(1, 11)

# Loop through different values of k
for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans_temp.fit(df_scaled)
    wcss.append(kmeans_temp.inertia_) # inertia_ holds the WCSS value

plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, marker='o', linestyle='-', color='purple', linewidth=2)
plt.title('The Elbow Method for Optimal k', fontsize=16)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('WCSS (Inertia)', fontsize=12)
plt.axvline(x=3, color='red', linestyle='--', label='Optimal k = 3')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Analysis: The graph drops rapidly from k=1 to k=3. After k=3, the reduction in WCSS is marginal (almost flat).")
print("This 'elbow' mathematically confirms that 3 is the optimal number of segments for this data.")

## 12. Core Concept 4: Validation - Silhouette Analysis
Another powerful validation metric is the **Silhouette Score**. 

It measures how similar an object is to its own cluster (cohesion) compared to other clusters (separation). The score ranges from -1 to +1:
- Near +1 indicates points are perfectly assigned to clusters.
- Near 0 indicates overlapping clusters.
- Negative values indicate misclassification.

In [ ]:
silhouette_scores = []
k_range_sil = range(2, 7) # Silhouette requires at least 2 clusters

for k in k_range_sil:
    kmeans_temp = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=42)
    temp_labels = kmeans_temp.fit_predict(df_scaled)
    # Calculate the average silhouette score across all samples
    score = silhouette_score(df_scaled, temp_labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
sns.barplot(x=list(k_range_sil), y=silhouette_scores, palette='magma')
plt.title('Silhouette Scores by Cluster Count', fontsize=14)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.axhline(y=max(silhouette_scores), color='red', linestyle='--')
plt.show()

print(f"The highest silhouette score is achieved at k=3 ({max(silhouette_scores):.3f}).")
print("This is a very high score, indicating excellent cluster density and separation.")

## 13. Core Concept 5: Hierarchical Clustering and Dendrograms
Unlike K-Means (Partitioning), Hierarchical Clustering builds a "tree" of relationships. You do not need to specify 'k' in advance.

**Agglomerative (Bottom-Up) Approach:**
1. Start with every data point as its own cluster.
2. Find the two closest clusters and merge them.
3. Repeat until all points are grouped into one massive cluster.

We visualize this using a **Dendrogram**, which helps us decide where to "cut" the tree to form our final segments.

In [ ]:
plt.figure(figsize=(15, 7))
plt.title('Dendrogram for Hierarchical Clustering', fontsize=16)
plt.xlabel('Physicians (Data Points)', fontsize=12)
plt.ylabel('Euclidean Distances', fontsize=12)

# Create the linkage matrix using 'ward' variance minimization algorithm
linkage_matrix = sch.linkage(df_scaled, method='ward')

# Plot the dendrogram
# For clarity, we truncate the plot to show only the last few merges
dendrogram = sch.dendrogram(
    linkage_matrix, 
    truncate_mode='level', 
    p=5,  # show only the last 5 levels of merges
    show_leaf_counts=True
)

# Draw a line to represent where we might 'cut' the tree to get 3 clusters
plt.axhline(y=15, color='red', linestyle='--', label='Cut Line for k=3')
plt.legend()
plt.show()

print("Reading the Dendrogram: The y-axis represents the distance between clusters when they merged.")
print("Long vertical lines indicate large distances between merging clusters. A cut across the three longest vertical branches yields 3 distinct clusters.")

## 14. Applying Agglomerative Clustering
Now that the dendrogram has suggested 3 clusters, let's fit the formal Agglomerative model and compare it to K-Means.

In [ ]:
# Initialize Agglomerative Clustering with 3 clusters
hc_model = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')

# Fit and predict
hc_labels = hc_model.fit_predict(df_scaled)

df_physicians['HC_Label'] = hc_labels

# Let's see if HC and K-Means agreed on the assignments
crosstab = pd.crosstab(df_physicians['Cluster_Label'], df_physicians['HC_Label'], 
                       rownames=['K-Means'], colnames=['Hierarchical'])

print("--- Agreement Matrix (K-Means vs Hierarchical) ---")
print(crosstab)
print("\nBecause the off-diagonal counts are 0, both algorithms perfectly agreed on the grouping of all 300 physicians.")
print("This is a testament to the strong natural structure in our data.")

## 15. Practice Exercises: Patient Adherence Segmentation
**Scenario:** You are analyzing a new cohort of 200 patients taking a chronic disease medication. You have data on their `Age`, `Dosage_Level` (mg), and an `Adherence_Score` (0-100 scale, measuring how consistently they take the drug).

We will generate the practice data.

In [ ]:
# Generate practice data for Patients
n_pat = 200

# Segment 1: Elderly, Low Dose, High Adherence
age_1 = np.random.normal(70, 5, 80)
dose_1 = np.random.normal(10, 2, 80)
adh_1 = np.random.normal(90, 5, 80)

# Segment 2: Young, High Dose, Low Adherence
age_2 = np.random.normal(30, 6, 120)
dose_2 = np.random.normal(50, 8, 120)
adh_2 = np.random.normal(45, 10, 120)

age_pat = np.clip(np.concatenate([age_1, age_2]), 18, 90)
dose_pat = np.clip(np.concatenate([dose_1, dose_2]), 5, 100)
adh_pat = np.clip(np.concatenate([adh_1, adh_2]), 0, 100)

df_patients = pd.DataFrame({
    'Age': age_pat,
    'Dosage_mg': dose_pat,
    'Adherence_Score': adh_pat
})

print("Practice Patient Dataset Created.")
print(df_patients.head())

### Exercise 1: Standardize and run the Elbow Method
1. Standardize the `df_patients` data.
2. Run a loop from k=1 to k=8 to calculate WCSS.
3. Plot the Elbow Curve.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
# 1. Standardize
scaler_pat = StandardScaler()
scaled_pat = scaler_pat.fit_transform(df_patients)

# 2. Calculate WCSS for Elbow Method
wcss_pat = []
for k in range(1, 9):
    kmeans_pat = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans_pat.fit(scaled_pat)
    wcss_pat.append(kmeans_pat.inertia_)

# 3. Plot
plt.figure(figsize=(8, 4))
plt.plot(range(1, 9), wcss_pat, marker='o', color='teal')
plt.title('Elbow Curve for Patient Data')
plt.xlabel('k')
plt.ylabel('WCSS')
plt.axvline(x=2, color='red', linestyle='--')
plt.show()

print("The elbow clearly occurs at k=2, indicating two primary patient segments.")

### Exercise 2: K-Means and Profiling
1. Fit K-Means with k=2 to the scaled patient data.
2. Add the cluster labels to the original `df_patients` dataframe.
3. Group by the labels and print the mean to profile the segments.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
# 1. Fit Model
kmeans_final_pat = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
pat_labels = kmeans_final_pat.fit_predict(scaled_pat)

# 2. Attach Labels
df_patients['Patient_Segment'] = pat_labels

# 3. Profile Segments
pat_profiles = df_patients.groupby('Patient_Segment').mean().round(1)
print("--- Patient Segment Profiles ---")
print(pat_profiles)

print("\nBusiness Action: Segment 1 (Younger, High Dose) has critically low adherence (~45/100).")
print("Intervention strategy (e.g., app reminders or SMS follow-ups) should be targeted entirely at Segment 1.")

## 16. Comparison Matrix and Key Takeaways

| Feature | Regression | Factor Analysis | Clustering |
|---|---|---|---|
| **Primary Goal** | Prediction | Data Reduction | Segmentation |
| **Target** | Dependent Variable (Y) | Latent Factors | Observation Groups |
| **Data Type** | Supervised | Unsupervised | Unsupervised |
| **Business Use** | Forecasting demand | Defining market drivers | Identifying target segments |

### Key Takeaways
1. **Preprocessing is Mandatory:** Because clustering relies on mathematical distance, you MUST standardize your data (Z-scores) so variables with large numbers don't overwhelm smaller ones.
2. **The K-Means Algorithm:** Iteratively updates centroids to maximize intra-cluster similarity and inter-cluster dissimilarity. 
3. **Validation:** The Elbow Method (inertia/WCSS) and Silhouette Scores provide statistical backing for choosing 'k'.
4. **Hierarchical Clustering:** Builds a tree (Dendrogram) showing the nested relationships of clusters, eliminating the need to guess 'k' upfront.
5. **Business Translation:** The true value of clustering is not the algorithm, but the strategic profiling of the resulting segments to enable targeted actions.

In [ ]:
print("Module 10: Cluster Analysis completed successfully!")
print("You are now ready to tackle unsupervised segmentation problems in the real world.")